# Fase 1: EDA y Baseline

En este notebook realizaremos la descarga de los datos de BreastMNIST y configuraremos el pipeline inicial de TensorFlow.

In [ ]:
import os
from medmnist import BreastMNIST

# Definimos la ruta apuntando a la carpeta de datos crudos creada previamente
base_dir = os.path.join('..', 'data', 'raw', 'breastmnist')
splits = ['train', 'val', 'test']

# Crear estructura de carpetas por clase (0: maligno, 1: benigno/normal)
for split in splits:
    for class_id in ['0', '1']:
        os.makedirs(os.path.join(base_dir, split, class_id), exist_ok=True)

def save_medmnist_to_disk(split_name):
    print(f"Procesando partición: {split_name}...")
    # Se exige usar la resolución nativa de 28x28 para el baseline
    dataset = BreastMNIST(split=split_name, download=True, size=28)
    
    for i in range(len(dataset)):
        img, label = dataset[i]
        class_label = str(label[0])
        
        img_path = os.path.join(base_dir, split_name, class_label, f"{split_name}_{i}.jpg")
        img.save(img_path)

# Ejecutar para las 3 particiones
# save_medmnist_to_disk('train') # Ejecutado por CLI
# save_medmnist_to_disk('val')
# save_medmnist_to_disk('test')
print("¡Imágenes listas en data/raw!")

In [ ]:
import tensorflow as tf
import os

BATCH_SIZE = 32
IMG_SIZE = (28, 28) # Resolución obligatoria para la Fase 1
base_dir = os.path.join('..', 'data', 'raw', 'breastmnist')

def create_tf_dataset(split_name):
    dir_path = os.path.join(base_dir, split_name)
    dataset = tf.keras.utils.image_dataset_from_directory(
        dir_path,
        shuffle=(split_name == 'train'), # Solo barajamos el entrenamiento
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        color_mode='rgb' # Requerido para modelos de Transfer Learning (3 canales)
    )
    # Optimización de rendimiento
    AUTOTUNE = tf.data.AUTOTUNE
    return dataset.cache().prefetch(buffer_size=AUTOTUNE)

# Instanciamos los datasets
train_dataset = create_tf_dataset('train')
val_dataset = create_tf_dataset('val')
test_dataset = create_tf_dataset('test')

print("¡Datasets de TensorFlow configurados y optimizados!")